<a href="https://colab.research.google.com/github/NikitaIvagin/ml-portfolio/blob/main/nlp/gpt_headline_generator/GPT_headline_generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets accelerate evaluate sacrebleu peft --quiet

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, TaskType
import torch
import numpy as np
from torch.nn.utils.rnn import pad_sequence
import random

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.5 MB/s eta 0:00:00


In [ ]:
dataset = load_dataset('IlyaGusev/gazeta')
sampled_dataset = {
    'train': dataset['train'].shuffle(seed=42).select(range(len(dataset['train']) // 5)), # Уменьшение, чтобы вписаться в ограничения колаба
    'validation': dataset['validation'].shuffle(seed=42).select(range(len(dataset['validation']) // 100))
}

model_checkpoint = "ai-forever/rugpt3small_based_on_gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
tokenizer.pad_token = tokenizer.eos_token

MAX_LENGTH = 512
SEPARATOR = "\n\nЗаголовок: "

def preprocess_function(examples):
    input_ids_batch = []
    labels_batch = []

    for news, title in zip(examples['summary'], examples['title']):
        # Новость (без добавления спецтокенов, чтобы не появились BOS/EOS)
        news_tokens = tokenizer(news, add_special_tokens=False, truncation=True, max_length=MAX_LENGTH)['input_ids']
        # Разделитель
        sep_tokens = tokenizer(SEPARATOR, add_special_tokens=False)['input_ids']
        # Заголовок
        title_tokens = tokenizer(title, add_special_tokens=False, truncation=True, max_length=MAX_LENGTH)['input_ids']

        input_ids = news_tokens + sep_tokens + title_tokens
        # Обрезаем по MAX_LENGTH (если превышает)
        if len(input_ids) > MAX_LENGTH:
            input_ids = input_ids[:MAX_LENGTH]

        # Сколько токенов ушло на новость + разделитель
        prefix_len = len(news_tokens) + len(sep_tokens)
        # Заголовок может быть обрезан, если общая длина > MAX_LENGTH
        title_len = len(title_tokens)
        # Реальная длина заголовка в финальной последовательности (может быть обрезан)
        actual_title_len = min(title_len, MAX_LENGTH - prefix_len)
        if actual_title_len < 0:
            actual_title_len = 0
            # Если новость+разделитель уже превысили MAX_LENGTH, заголовок не помещается — тогда весь labels = -100
            labels = [-100] * len(input_ids)
        else:
            labels = [-100] * prefix_len + title_tokens[:actual_title_len]
            # Если input_ids оказался длиннее prefix_len+actual_title_len (редко, из-за обрезки),
            # то лишние токены (если есть) помечаем -100
            if len(input_ids) > len(labels):
                labels += [-100] * (len(input_ids) - len(labels))
            elif len(labels) > len(input_ids):
                labels = labels[:len(input_ids)]

        input_ids_batch.append(input_ids)
        labels_batch.append(labels)

    return {'input_ids': input_ids_batch, 'labels': labels_batch}

tokenized_train = sampled_dataset['train'].map(preprocess_function, batched=True, remove_columns=sampled_dataset['train'].column_names)
tokenized_eval = sampled_dataset['validation'].map(preprocess_function, batched=True, remove_columns=sampled_dataset['validation'].column_names)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

default/train/0000.parquet:   0%|          | 0.00/252M [00:00<?, ?B/s]

default/train/0001.parquet:   0%|          | 0.00/22.7M [00:00<?, ?B/s]

default/validation/0000.parquet:   0%|          | 0.00/27.8M [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/574 [00:00<?, ?B/s]

Map:   0%|          | 0/12192 [00:00<?, ? examples/s]

Map:   0%|          | 0/63 [00:00<?, ? examples/s]

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_checkpoint,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)
model.config.pad_token_id = tokenizer.pad_token_id

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["c_attn", "c_proj", "c_fc"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/551M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/551M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: ai-forever/rugpt3small_based_on_gpt2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 
transformer.h.{0...11}.attn.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


trainable params: 1,179,648 || all params: 165,014,016 || trainable%: 0.7149


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [ ]:
class DynamicPadCollatorForCausalLM:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, batch):
        input_ids = [torch.tensor(item['input_ids'], dtype=torch.long) for item in batch]
        labels = [torch.tensor(item['labels'], dtype=torch.long) for item in batch]

        input_ids_padded = pad_sequence(input_ids, batch_first=True, padding_value=self.tokenizer.pad_token_id)
        labels_padded = pad_sequence(labels, batch_first=True, padding_value=-100)

        attention_mask = (input_ids_padded != self.tokenizer.pad_token_id).long()

        return {
            'input_ids': input_ids_padded,
            'attention_mask': attention_mask,
            'labels': labels_padded,
        }

data_collator = DynamicPadCollatorForCausalLM(tokenizer)

In [ ]:
training_args = TrainingArguments(
    output_dir="./gpt_news_title",
    do_eval=True,
    learning_rate=2e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    eval_steps=len(tokenized_train) // 4,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=1,
    logging_dir='./logs',
    logging_steps=500,
    fp16=torch.cuda.is_available(),
    remove_unused_columns=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
trainer.train()

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
500,3.293542
1000,3.119114
1500,3.120103
2000,3.104577
2500,3.117022
3000,3.100463


TrainOutput(global_step=3048, training_loss=3.141384735507915, metrics={'train_runtime': 258.0225, 'train_samples_per_second': 47.252, 'train_steps_per_second': 11.813, 'total_flos': 922047434035200.0, 'train_loss': 3.141384735507915, 'epoch': 1.0})

In [ ]:
def generate_title(news_text, model, tokenizer, max_new_tokens=50):
    prompt = news_text.strip() + SEPARATOR
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_LENGTH)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_k=50,
        num_beams=1,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id
    )
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if SEPARATOR in full_output:
        title = full_output.split(SEPARATOR, 1)[-1].strip()
        # Обрезаем по первому двоеточию для улучшения читаемости
        if ':' in title:
            title = title.split(':', 1)[0].strip()
    else:
        title = full_output.strip()
    return title

In [ ]:
# Выбираем случайный пример из валидационной выборки
example_index = random.randint(0, len(sampled_dataset['validation']) - 1)
example = sampled_dataset['validation'][example_index]

news_summary = example['summary']
true_title = example['title']

print(f"Оригинальный текст новости:\n{news_summary}")
print(f"\nИстинный заголовок:\n{true_title}")

predicted_title = generate_title(news_summary, model, tokenizer)
print(f"\nСгенерированный моделью заголовок:\n{predicted_title}")

Оригинальный текст новости:
Президент Франции Эммануэль Макрон впервые приехал в Польшу с официальным визитом. В условиях передела сил в Европе после Brexit переговоры Варшавы с Парижем имеют особую важность — отстаивающему интересы ЕС Макрону предстоит разрешить противоречия с одним из самых своеобразных и независимых членов европейского объединения.

Истинный заголовок:
После Brexit: Франция и Польша начинают делить Европу

Сгенерированный моделью заголовок:
Отношения с ЕС и Макроном
